<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/12WSPOLBIEZNOSC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_wspolbieznosc.py

from __future__ import annotations

import asyncio
import json

from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Final

import aiohttp

from modul_pobieranieW import (
    TwelveDataAdapter,
)


FOLDER_DANYCH: Final[Path] = Path(
    "/content/drive/MyDrive/projekt_test"
)

TWELVE_DATA_URL: Final[str] = (
    "https://api.twelvedata.com/time_series"
)

TIMEOUT_REQUEST: Final[float] = 30.0

MAX_CONCURRENT_REQUESTS: Final[int] = 5


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True,
)
class WynikPobieraniaAsync:

    ticker: str

    sukces: bool

    dane: dict[str, Any] | None = field(
        default=None,
        compare=False,
    )

    blad: str | None = field(
        default=None,
        compare=False,
    )


class AsyncStockApiGateway:

    def __init__(
        self,
        klucz: str,
        *,
        timeout: float = TIMEOUT_REQUEST,
        max_concurrent: int = MAX_CONCURRENT_REQUESTS,
    ) -> None:

        if not klucz:
            raise ValueError(
                "klucz API nie moze byc pusty"
            )

        if timeout <= 0:
            raise ValueError(
                "timeout musi byc wiekszy od 0"
            )

        if max_concurrent <= 0:
            raise ValueError(
                "max_concurrent musi byc wieksze od 0"
            )

        self.klucz = klucz

        self.timeout = timeout

        self._semaphore = asyncio.Semaphore(
            max_concurrent
        )

    async def pobierz_json(
        self,
        session: aiohttp.ClientSession,
        ticker: str,
    ) -> dict[str, Any]:

        ticker = ticker.strip().upper()

        if not ticker:
            raise ValueError(
                "ticker nie moze byc pusty"
            )

        parametry: dict[str, Any] = {
            "symbol": ticker,
            "interval": "1day",
            "outputsize": 5000,
            "apikey": self.klucz,
        }

        async with self._semaphore:

            async with session.get(
                TWELVE_DATA_URL,
                params=parametry,
            ) as response:

                response.raise_for_status()

                dane: Any = await response.json()

        if not isinstance(
            dane,
            dict,
        ):
            raise ValueError(
                f"{ticker}: odpowiedz API "
                "nie jest obiektem JSON"
            )

        return dane


async def pobierz_jeden_ticker(
    gateway: AsyncStockApiGateway,
    session: aiohttp.ClientSession,
    ticker: str,
) -> WynikPobieraniaAsync:

    ticker = ticker.strip().upper()

    try:

        dane: dict[str, Any] = (
            await asyncio.wait_for(
                gateway.pobierz_json(
                    session,
                    ticker,
                ),
                timeout=gateway.timeout,
            )
        )

        return WynikPobieraniaAsync(
            ticker=ticker,
            sukces=True,
            dane=dane,
        )

    except asyncio.TimeoutError:

        return WynikPobieraniaAsync(
            ticker=ticker,
            sukces=False,
            blad=(
                f"timeout podczas pobierania "
                f"{ticker}"
            ),
        )

    except Exception as e:

        return WynikPobieraniaAsync(
            ticker=ticker,
            sukces=False,
            blad=str(e),
        )


async def pobierz_wiele_tickerow(
    gateway: AsyncStockApiGateway,
    tickery: list[str],
) -> list[WynikPobieraniaAsync]:

    timeout_http = aiohttp.ClientTimeout(
        total=gateway.timeout
    )

    async with aiohttp.ClientSession(
        timeout=timeout_http
    ) as session:

        taski: list[
            asyncio.Task[WynikPobieraniaAsync]
        ] = []

        for ticker in tickery:

            coroutine = pobierz_jeden_ticker(
                gateway,
                session,
                ticker,
            )

            task = asyncio.create_task(
                coroutine
            )

            taski.append(
                task
            )

        surowe_wyniki: list[
            WynikPobieraniaAsync | BaseException
        ] = await asyncio.gather(
            *taski,
            return_exceptions=True,
        )

    wyniki: list[
        WynikPobieraniaAsync
    ] = []

    for ticker, wynik in zip(
        tickery,
        surowe_wyniki,
    ):

        if isinstance(
            wynik,
            BaseException,
        ):

            wyniki.append(
                WynikPobieraniaAsync(
                    ticker=ticker,
                    sukces=False,
                    blad=str(wynik),
                )
            )

        else:

            wyniki.append(
                wynik
            )

    return wyniki


def zapisz_json(
    ticker: str,
    dane: dict[str, Any],
    folder: Path = FOLDER_DANYCH,
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    plik: Path = (
        folder
        / f"{ticker}_twelve.json"
    )

    with open(
        plik,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return plik


def przetworz_i_zapisz(
    wyniki: list[WynikPobieraniaAsync],
    folder: Path = FOLDER_DANYCH,
) -> list[Path]:

    adapter = TwelveDataAdapter()

    zapisane: list[Path] = []

    for wynik in wyniki:

        if not wynik.sukces:

            print(
                wynik.ticker,
                "-> BLAD:",
                wynik.blad,
            )

            continue

        if wynik.dane is None:

            print(
                wynik.ticker,
                "-> brak danych"
            )

            continue

        try:

            plik_json: Path = (
                zapisz_json(
                    wynik.ticker,
                    wynik.dane,
                    folder,
                )
            )

            historia = (
                adapter.response_to_historia(
                    wynik.ticker,
                    wynik.dane,
                )
            )

            plik_csv: Path = (
                folder
                / f"{wynik.ticker}_twelve.csv"
            )

            historia.to_dataframe().to_csv(
                plik_csv,
                index=False,
            )

            zapisane.append(
                plik_json
            )

            zapisane.append(
                plik_csv
            )

            print(
                wynik.ticker,
                "-> OK"
            )

            print(
                "JSON:",
                plik_json,
            )

            print(
                "CSV:",
                plik_csv,
            )

        except Exception as e:

            print(
                wynik.ticker,
                "-> blad przetwarzania:",
                e,
            )

    return zapisane


async def main_async(
    tickery: list[str],
    klucz_api: str,
) -> list[WynikPobieraniaAsync]:

    gateway = AsyncStockApiGateway(
        klucz=klucz_api,
        timeout=30.0,
        max_concurrent=5,
    )

    wyniki: list[
        WynikPobieraniaAsync
    ] = await pobierz_wiele_tickerow(
        gateway,
        tickery,
    )

    return wyniki


def run() -> None:

    tekst: str = input(
        "podaj tickery oddzielone przecinkami: "
    )

    tickery: list[str] = [
        ticker.strip().upper()

        for ticker
        in tekst.split(",")

        if ticker.strip()
    ]

    if not tickery:

        raise ValueError(
            "nie podano tickerow"
        )

    klucz_api: str = input(
        "podaj klucz Twelve Data API: "
    ).strip()

    if not klucz_api:

        raise ValueError(
            "nie podano klucza API"
        )

    print(
        "\nSTART ASYNC"
    )

    print(
        "tickery:",
        tickery,
    )

    wyniki: list[
        WynikPobieraniaAsync
    ] = asyncio.run(
        main_async(
            tickery,
            klucz_api,
        )
    )

    print(
        "\nPOBIERANIE ZAKONCZONE"
    )

    sukcesy: int = sum(
        1
        for wynik in wyniki
        if wynik.sukces
    )

    bledy: int = (
        len(wyniki)
        - sukcesy
    )

    print(
        "sukcesy:",
        sukcesy,
    )

    print(
        "bledy:",
        bledy,
    )

    print(
        "\nDALSZE PRZETWARZANIE "
        "JUZ SYNCHRONICZNE"
    )

    zapisane: list[Path] = (
        przetworz_i_zapisz(
            wyniki
        )
    )

    print(
        "\nLICZBA ZAPISANYCH PLIKOW:",
        len(zapisane),
    )

    print(
        "\nMODUL WSPOLBIEZNOSCI "
        "DZIALA POPRAWNIE"
    )